# Delete single-band TIFF artifacts

Removes leftover per-band GeoTIFFs that were copied into the resampled output tree (stems ending in band tags like `.r`, `.g`, `.b`, `.nir`, or `.sqr1` / `.sqr2` / `.sqr<N>`).

**Default target:** `PROJECT_ROOT / "data" / "sat_images"` (same layout as `convert_sites_batch.ipynb`). Change `TARGET_ROOT` if your outputs live under another folder (for example a `track` tree).

1. Run the config + scan cells and confirm the listed paths.
2. Set `DELETE_FILES = True` and run the delete cell.

In [5]:
from __future__ import annotations

import re
from pathlib import Path

try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd()

# Resampled outputs from this repo (edit if needed).
TARGET_ROOT = PROJECT_ROOT / "data" / "sat_images"
# Example: TARGET_ROOT = PROJECT_ROOT / "track" / "sat_images"

# Exact stem suffixes (before .tif / .tiff), compared case-insensitively.
BAND_SUFFIXES: tuple[str, ...] = (
    ".r",
    ".g",
    ".b",
    ".nir",
    ".swir1",
    ".swir2",
)

# Any .sqr followed by digits (e.g. .sqr3).
_SQR_DIGITS = re.compile(r"\.sqr\d+$", re.IGNORECASE)


def is_band_artifact_tiff(path: Path) -> bool:
    if path.suffix.lower() not in {".tif", ".tiff"}:
        return False
    stem_lower = path.stem.lower()
    for suf in BAND_SUFFIXES:
        if stem_lower.endswith(suf.lower()):
            return True
    return bool(_SQR_DIGITS.search(stem_lower))


def collect_artifact_tiffs(root: Path) -> list[Path]:
    if not root.is_dir():
        return []
    found: list[Path] = []
    for pattern in ("**/*.tif", "**/*.tiff"):
        for p in root.glob(pattern):
            if is_band_artifact_tiff(p):
                found.append(p)
    return sorted(found)


print("PROJECT_ROOT:", PROJECT_ROOT)
print("TARGET_ROOT:", TARGET_ROOT.resolve())

PROJECT_ROOT: c:\Users\jnicolow\Documents\research\CRC\change_tiff_resolution
TARGET_ROOT: C:\Users\jnicolow\Documents\research\CRC\change_tiff_resolution\data\sat_images


In [6]:
candidates = collect_artifact_tiffs(TARGET_ROOT)
print(f"Found {len(candidates)} file(s) matching band-artifact name patterns.")
for p in candidates[:50]:
    print(p)
if len(candidates) > 50:
    print(f"... and {len(candidates) - 50} more")

Found 5260 file(s) matching band-artifact name patterns.
c:\Users\jnicolow\Documents\research\CRC\change_tiff_resolution\data\sat_images\australianarrabeen\bicubic\L5\L5_LT05_089083_19870522.SWIR1.tif
c:\Users\jnicolow\Documents\research\CRC\change_tiff_resolution\data\sat_images\australianarrabeen\bicubic\L5\L5_LT05_089083_19870522.SWIR2.tif
c:\Users\jnicolow\Documents\research\CRC\change_tiff_resolution\data\sat_images\australianarrabeen\bicubic\L5\L5_LT05_089083_19870911.SWIR1.tif
c:\Users\jnicolow\Documents\research\CRC\change_tiff_resolution\data\sat_images\australianarrabeen\bicubic\L5\L5_LT05_089083_19870911.SWIR2.tif
c:\Users\jnicolow\Documents\research\CRC\change_tiff_resolution\data\sat_images\australianarrabeen\bicubic\L5\L5_LT05_089083_19870927.SWIR1.tif
c:\Users\jnicolow\Documents\research\CRC\change_tiff_resolution\data\sat_images\australianarrabeen\bicubic\L5\L5_LT05_089083_19870927.SWIR2.tif
c:\Users\jnicolow\Documents\research\CRC\change_tiff_resolution\data\sat_images

In [8]:
# Set True only after reviewing the list above.
DELETE_FILES = True

if not DELETE_FILES:
    print("Set DELETE_FILES = True to remove files.")
else:
    removed = 0
    for p in candidates:
        p.unlink()
        removed += 1
    print(f"Deleted {removed} file(s).")

Deleted 5260 file(s).
